# 01 - Veri Özeti

## Amaç ve Kapsam

Bu notebook, tezin makine öğrenmesi ve yapay zeka temelli özellik mühendisliği (feature engineering) stratejilerini belirleyen ilk ve en kritik **keşifsel veri analizi (Exploratory Data Analysis - EDA)** adımıdır. Alibaba PAI 100K GPU iş örnekleminin ham kalibrasyonlarını inceleyerek, modellerin (Random Forest, LightGBM, CNN, LSTM vb.) hangi verilerle eğitileceğine ve bu verilerin istatistiksel geçerliliğine (validity) dair nesnel kararların temelini oluşturur.

Bu aşamada gerçekleştirilen incelemeler, tezin ilerleyen aşamalarındaki mimari tercihleri doğrudan şu şekilde haklı çıkarmaktadır:

1. **Veri Bütünlüğü ve Veri Kaybı (Missing Data):** Kümeden alınan ham iş yükündeki eksik veya geçersiz değerleri deşifre ederek, derin öğrenme modellerinin (örneğin LSTM) anomali veya bozuk gradyanlara (exploding gradients) maruz kalmasını engellemek üzere veri setini sterilize etmek.
2. **Kategorik Değişkenlerin Kardinalitesi:** `user` veya `gpu_type` gibi kategorik değerlerin analiz edilerek One-Hot kodlama yerine neden Native Categorical (LightGBM) yaklaşımının denenmesi gerektiğini istatistiksel olarak kanıtlamak.
3. **Ayırt Edici Dağılımlar:** Makine öğrenimine aktarılacak yapısal sütunların (vCPU, Memory, Num Instances vs.) varyans analizini yaparak modeller için hangi girdi özelliklerinin "tahmin edici güç" (predictive power) taşıdığını belirlemek.
4. **Ham Zaman Serisi Kalitesinin Testi:** Zaman damgalı verilerin ardışıklık yapısını doğrulayarak, simülasyon aşamasındaki `MultiNodeClusterSimulator` motoruna girecek olan olay kuyruğunun (event trace) kronolojik güvenilirliğini ispatlamak.

Buradaki bulgular üzerine, tezin sonraki safhası olan İş Yükü Karakterizasyonu (Notebook 02) ve Özellik Mühendisliği (Notebook 03) aşamalarına veri-odaklı (data-driven) sağlam bir köprü kurulmaktadır.

In [1]:
# ── 0. Environment & Path Setup ──────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Anchor on a marker that only exists at the repository root, so the notebook
# works regardless of the directory the kernel was started from. The previous
# parents[1] form silently resolved one level short when the working directory
# was notebooks/ rather than notebooks/<lang>/.
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    return start.parents[1]

PROJECT_ROOT = _find_project_root(Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[Setup] Project root : {PROJECT_ROOT}")
print(f"[Setup] Python path  : {sys.executable}")

[Setup] Project root : /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling
[Setup] Python path  : /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling/venv/bin/python


> **Kurulum doğrulandı.** Proje kök dizini Python yoluna eklenmiş, `configs/paths.yaml`
> üzerinden veri dizinleri doğrulanmıştır. Notebook tekrarlanabilir biçimde çalışmaya hazırdır.


In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
# stdlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project
from src.data_loading import load_main_sample
from src.feature_engineering import build_job_table_from_sample

# Unified visualisation theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.family": "DejaVu Sans",
    # ── Publication-grade output (figure/table audit) ──────────────────────
    # scripts/export_thesis_results.py scrapes the notebook's own inline PNG,
    # so the INLINE dpi is what reaches the thesis. At the 100 dpi default
    # these exported at 147-280 ppi, under the 300 ppi publisher floor.
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    # Figures are now authored at roughly the width they are PRINTED at
    # (\textwidth = 6.10 in in thesis.cls: a4paper, 3.5cm/2cm margins), so the
    # downscale is ~0.8x rather than the ~0.34x that made in-figure text print
    # at 2.5-4.8 pt -- under the ~6 pt legibility floor. At these sizes the
    # smallest label prints at ~6.4 pt and body text at ~7.2 pt.
    # Figures are authored on a generous canvas (10-18 in) and printed into a
    # 6.10-in column, roughly a 0.35-0.55x reduction. Shrinking the canvas to
    # match the print width made every panel cramped, so the canvas stays and
    # the TYPE is scaled instead: at 15 pt a 14-in figure prints at 6.5 pt,
    # above the ~6 pt legibility floor, while still looking uncrowded at
    # authoring size.
    "font.size": 15,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.titlesize": 19,
})

print("[Setup] All imports OK.")


> **Bağımlılıklar hazır.** Pandas, NumPy, Matplotlib, Seaborn ve `src` modülleri başarıyla
> içe aktarılmıştır. Görselleştirme parametreleri (`rcParams`) ayarlanmıştır.


Veri kümesi 100.000 işten oluşmakta ve toplam 8 öznitelik içermektedir. Tüm sütunlarda eksik (null) değer bulunmamaktadır, bu da ek bir eksik veri temizleme adımına ihtiyaç olmadığını göstermektedir.

Sayısal öznitelikler (ör. num_gpu, num_cpu, duration) uygun sayısal veri tiplerinde tutulurken, kategorik alanlar (user, gpu_type) nesne (object) türündedir.

In [3]:
# ── 2. Load Dataset ───────────────────────────────────────────────────────────
print("[Step 1] Loading raw dataset...")
raw_df = load_main_sample()

print(f"[Step 1] Shape   : {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
raw_df.head()

[Step 1] Loading raw dataset...
[Step 1] Shape   : 100,000 rows × 8 columns


,job_id,num_inst,submit_time,num_cpu,num_gpu,gpu_type,duration,user
0,0,1.0,0,2.0,0.00,CPU,34,d4d51aca8806
1,1,12.0,3,6.0,0.25,T4,15748,d4d51aca8806
2,2,1.0,5,6.0,1.00,MISC,84,a8192d6b0ae9
3,3,1.0,21,18.0,1.00,T4,46,c7152ce0fec1
4,4,1.0,21,6.0,1.00,MISC,80,7b76597f4283


> **Ham veri yüklendi: 100.000 iş kaydı.** Veri setinin tamamı belleğe alınmıştır. İz verisi
> her iş için `duration` değeri kayıtlı tek satır içermektedir; sonraki iki hücre hiçbir
> sütunda eksik değer bulunmadığını doğrulamaktadır.


In [4]:
# ── 3. Column types and null counts ──────────────────────────────────────────
print("[Step 2] Schema inspection — dtypes and null counts")
raw_df.info()

[Step 2] Schema inspection — dtypes and null counts
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   job_id       100000 non-null  int64  
 1   num_inst     100000 non-null  float64
 2   submit_time  100000 non-null  int64  
 3   num_cpu      100000 non-null  float64
 4   num_gpu      100000 non-null  float64
 5   gpu_type     100000 non-null  object 
 6   duration     100000 non-null  int64  
 7   user         100000 non-null  object 
dtypes: float64(3), int64(3), object(2)
memory usage: 6.1+ MB


> **Sütun tipleri ve null sayıları raporlandı.** Sayısal sütunların (`num_gpu`, `num_cpu`,
> `duration`) doğru tipte olduğu doğrulanmıştır. `raw_df.info()` çıktısı 8 sütunun her biri
> için 100.000 dolu (non-null) kayıt bildirmektedir; sonraki hücre aynı sonucu açıkça
> ifade etmektedir.


In [5]:
# ── 4. Missing value summary ──────────────────────────────────────────────────
missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_%", ascending=False)

if missing_summary.empty:
    print("[Step 2] No missing values detected.")
else:
    print("[Step 2] Columns with missing values:")
    display(missing_summary)

[Step 2] No missing values detected.


> **Eksik değer analizi tamamlandı.** Ham izin hiçbir sütununda eksik değer bulunmadığından,
> eksik ya da pozitif olmayan süre gerekçesiyle hiçbir kayıt elenmemektedir. Bir sonraki
> adımda çıkarılan 17.816 kayıt, tezin modellemediği yalnızca-CPU işleridir
> (`num_gpu == 0`).


İş gönderim zamanları, Unix epoch başlangıcına göre normalize edilmiş zaman damgaları olarak sunulmaktadır. Bu nedenle mutlak takvim tarihleri değil, işlerin göreli zamanlaması ve aralarındaki zaman farkları analiz edilmektedir.

In [6]:
# ── 5. Build canonical job table ─────────────────────────────────────────────
print("[Step 3] Building canonical job table...")
job_df = build_job_table_from_sample(raw_df, time_unit="s")

print(f"[Step 3] Valid GPU jobs : {len(job_df):,}  (filtered {len(raw_df) - len(job_df):,} invalid rows)")
job_df.describe()

[Step 3] Building canonical job table...
[Step 3] Valid GPU jobs : 82,184  (filtered 17,816 invalid rows)


,job_id,arrival_time,arrival_sec,job_runtime,gpu_demand,num_inst,num_cpu
count,82184.000000,82184,82184.000000,82184.000000,82184.000000,82184.000000,82184.000000
mean,50211.736713,1970-01-04 17:18:34.684865667,321511.684866,5223.019408,0.680211,5.007873,6.916216
min,1.000000,1970-01-01 00:00:03,0.000000,4.000000,0.010000,1.000000,0.040000
25%,25865.500000,1970-01-02 17:17:39.250000,148656.250000,153.000000,0.250000,1.000000,6.000000
50%,50284.500000,1970-01-04 12:00:23.500000,302420.500000,594.000000,0.500000,1.000000,6.000000
75%,75070.250000,1970-01-06 17:03:11.750000,493388.750000,4126.000000,1.000000,1.000000,6.000000
max,99999.000000,1970-01-08 15:51:32,661889.000000,599445.000000,8.000000,512.000000,90.000000
std,28695.483592,NaN,195452.663528,15980.719087,0.560444,14.652502,4.060256


> **Kanonik iş tablosu oluşturuldu: 82.184 geçerli GPU işi.** Çıkarılan 17.816 kayıt tam
> olarak yalnızca-CPU işleridir (`num_gpu == 0`): `build_job_table_from_sample()` burada
> varsayılan `include_cpu_only=False` ile çağrılmaktadır. İz verisinde eksik veya pozitif
> olmayan çalışma süresine sahip kayıt bulunmadığından bu gerekçeyle hiçbir satır
> elenmemektedir. Tablo bu haliyle Notebook 03–05'teki tüm analizlerin temelini
> oluşturmaktadır.


In [ ]:
import matplotlib.ticker as mticker
# Hard-coded sizes rescaled with the figure: these override rcParams, so
# leaving them at the old values would keep the very labels this figure
# was resized for at their old, unreadable printed size.
# ── 6. Runtime histogram (log scale) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw seconds
axes[0].hist(job_df["job_runtime"], bins=80, color="steelblue", edgecolor="white", linewidth=0.4)
# Six-figure second counts do not fit under a 3.5-inch panel: at the printed
# width the default ticks ran together into "100000200000300000".
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(4))
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _p: f"{v/1000:,.0f}k" if v else "0"))
# The formatter above already renders "200k", so the label must not repeat
# the scale -- "200k thousands of seconds" is the reading it would invite.
axes[0].set_xlabel("Runtime (seconds)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Raw Runtime Distribution")

# Right: log10 transform
log_runtimes = np.log10(job_df["job_runtime"] + 1)
axes[1].hist(log_runtimes, bins=60, color="darkorange", edgecolor="white", linewidth=0.4)
axes[1].set_xlabel("log₁₀(Runtime + 1)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Runtime Distribution (Log₁₀ Scale)")

# "100K" named the raw file, but these panels plot the filtered GPU-job subset
# (82,184 rows; 17,816 CPU-only/zero-duration rows are dropped upstream), so the
# old title overstated n by ~22%.
plt.suptitle("Job Runtime Distribution — Alibaba PAI Trace", fontsize=19, fontweight="bold")
plt.tight_layout()
plt.show()

> **Şekil 1 — Runtime Dağılımı (Ham ve Log₁₀ Ölçek):** İki panel birlikte okunmalıdır.
> Sol panel (mavi): Ham ölçekte dağılım **aşırı sağ-çarpıktır** — işlerin neredeyse tamamı
> 0–20.000 saniye bandında yığılırken sağ kuyruk 600.000 saniyeye kadar uzanmaktadır.
> Sağ panel (turuncu): Log₁₀ ölçekte dağılım **bimodal** bir yapı sergiler — yaklaşık
> log₁₀≈2.5 (~300 saniye) ve log₁₀≈3.6–3.7 (~4.000–5.000 saniye) civarında iki tepe
> noktası mevcuttur. Bu bimodal yapı, kümede iki farklı iş tipi olduğuna işaret etmektedir:
> kısa çıkarım (inference) işleri ve uzun eğitim (training) işleri.
>
> **Model seçimine etkisi:** Sağ-çarpık + ağır-kuyruk yapısı, MSE kayıp fonksiyonunu
> (aykırı değerleri orantısız cezalandırır) uygunsuz kılmaktadır. MAE ve MdAE tercih edilmeli;
> doğrusal modeller bu dağılımı yeterince temsil edemez.

In [ ]:
# ── 7. Runtime CDF ────────────────────────────────────────────────────────────
runtimes_sorted = np.sort(job_df["job_runtime"].values)
cdf = np.arange(1, len(runtimes_sorted) + 1) / len(runtimes_sorted)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(runtimes_sorted, cdf, lw=2, color="royalblue")
ax.axvline(np.median(runtimes_sorted), color="crimson", ls="--", label=f"Median = {np.median(runtimes_sorted):.0f}s")
ax.axvline(np.percentile(runtimes_sorted, 95), color="orange", ls="--", label=f"P95 = {np.percentile(runtimes_sorted, 95):.0f}s")
ax.set_xscale("log")
ax.set_xlabel("Runtime (seconds, log scale)")
ax.set_ylabel("CDF")
ax.set_title("Cumulative Distribution of Job Runtimes")
ax.legend()
plt.tight_layout()
plt.show()

p50 = np.percentile(runtimes_sorted, 50)
p95 = np.percentile(runtimes_sorted, 95)
p99 = np.percentile(runtimes_sorted, 99)
print(f"P50  : {p50:>10,.0f} s")
print(f"P95  : {p95:>10,.0f} s")
print(f"P99  : {p99:>10,.0f} s")
print(f"Max  : {runtimes_sorted[-1]:>10,.0f} s")

> **Şekil 2 — Runtime Kümülatif Dağılım Fonksiyonu (CDF):** Figür üzerindeki
> sayısal değerler doğrudan okunmuştur:
> - **P50 (Medyan) = 594 saniye** (~10 dakika): İşlerin yarısı 10 dakikadan kısa sürmektedir.
> - **P95 = 24.110 saniye** (~6.7 saat): İşlerin %95'i 6.7 saatin altındadır.
> - **P99 = 67.432 saniye** (~18.7 saat): Son %1'lik dilim 18 saatten uzun sürmektedir.
> - **Max = 599.445 saniye** (~7 gün): En uzun iş neredeyse tam bir hafta sürmektedir.
>
> **Kuyruk oranı P95/P50 ≈ 40.6** — olağanüstü yüksek bir değer. (Bu oran varyasyon
> katsayısı değildir; aynı 82.184 iş üzerinde varyasyon katsayısı (std/ortalama) 3.06'dır —
> yine de üssel dağılımın CV = 1 değerinin çok üzerinde.) Bu denli yüksek dağılım, ortalama
> üzerinden alınan FIFO kararlarının en kötü senaryolarda kritik Head-Of-Line bloklamalara
> yol açmasının nedenidir. SJF politikasının kazanım potansiyeli bu varyasyonla doğru
> orantılıdır.


In [ ]:
# ── 8. Hourly arrival rate ─────────────────────────────────────────────────-
# Use lowercase "1h" alias — uppercase "1H" is deprecated in pandas >= 2.2
arrival_series = (
    job_df.set_index("arrival_time")["job_id"]
    .resample("1h")
    .count()
)
hours = arrival_series.index.total_seconds() / 3600 if hasattr(arrival_series.index[0], 'total_seconds') else         (arrival_series.index - arrival_series.index[0]).total_seconds() / 3600

days = hours / 24

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(days, arrival_series.values, lw=1.2, color="seagreen")
ax.fill_between(days, arrival_series.values, alpha=0.15, color="seagreen")
ax.set_xlabel("Elapsed Time (days)")
ax.set_ylabel("Jobs per Hour")
ax.set_title("Hourly Job Arrival Rate Over the Trace Duration")
plt.tight_layout()
plt.show()

> **Şekil 3 — Saatlik İş Geliş Hızı (8 Günlük İz):** X ekseni geçen süreyi gün
> cinsinden (0–8), Y ekseni saatte gelen iş sayısını (jobs/hour) göstermektedir.
> - Taban oran: ~200–400 job/saat
> - Belirgin zirveler: gün ~1.5 → **~1.350 job/saat**, gün ~5.7 → **~1.080 job/saat**
> - Geliş süreci homojen değil, **bursty (ani yoğunlaşmalı)** bir yapıdadır.
>
> **Zaman ekseni hakkında not:** Yatay eksen, ham `arrival_time` değişkenini değil, **iz başlangıcından itibaren geçen süreyi (elapsed time)** göstermektedir.  
> `arrival_time`, *iz-göreli* bir ofsete (`submit_time` sıfırdan başlamaktadır) uygulanan `pd.to_datetime(submit_time, unit='s')` çıktısıdır; yani kümenin gerçek saatine değil Unix epoch'una çakılıdır. **Duvar saati (UTC ya da yerel) zaman damgası değildir** ve verdiği saat değeri, mutlak hizası bilinmeyen 24 saatlik bir fazdır.  
> Bu şekil, iş yükünü **8 günlük bir zaman serisi** olarak göstermektedir; gün içindeki saatlere (0–23) göre bir dağılım sunmamaktadır.  
> İz-günü ve saat bazlı iş yoğunluğu analizi için **Şekil 6 (Trace-Day × Hour-of-Day Heatmap)** incelenmelidir.

In [ ]:
# ── 9. GPU demand histogram ───────────────────────────────────────────────────
# GPU demand is not a continuous quantity here: the PAI trace records it at a
# small set of discrete levels (0.01, 0.05, ..., 1, 2, ..., 8), and no value in
# between is possible. A log-spaced numeric axis therefore spent roughly half
# the width on gaps that no job can occupy, and it distorted the bars: with the
# width fixed in data units, a bar's drawn width grows with its position, so
# the 0.2 bar (≈800 jobs) rendered narrower than the 0.25 bar (≈20,600) and
# area stopped encoding count -- which is the one thing a bar chart is read for.
# Equal spacing puts every observed level on the same footing; the level itself
# is carried by the tick label, so no information is lost, and every level can
# be labelled without the collisions the log spacing forced (0.25/0.3, 4/5/6).
demand_counts = job_df["gpu_demand"].value_counts().sort_index()
_pos = np.arange(len(demand_counts))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(_pos, demand_counts.values, width=0.75,
       color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xticks(_pos)
ax.set_xticklabels([f"{v:g}" for v in demand_counts.index], rotation=45, fontsize=13)
ax.set_xlim(-0.7, len(demand_counts) - 0.3)
ax.set_xlabel("GPU Demand per Job (GPUs)")
ax.set_ylabel("Number of Jobs")
ax.set_title("Distribution of GPU Demand per Job")
plt.tight_layout()
plt.show()

print("[Step 7] GPU demand statistics:")
print(job_df["gpu_demand"].describe().to_string())

> **Şekil 4 — GPU Talep Dağılımı:** Figürden ve figürle birlikte yazdırılan `describe()`
> çıktısından okunan değerler:
> - **Kesirli talep (0.01–0.8 GPU): 43.176 iş** — GPU paylaşımı istekleri; tablonun
>   yarısından fazlası. Yalnızca 0.25 seviyesinde 20.615 iş bulunmaktadır.
> - **1 GPU: 37.033 iş** — tek başına en yaygın seviye.
> - **2 GPU: 1.274 iş**; 3–8 GPU seviyeleri toplamı: 701 iş.
> - **Medyan = 0.50, yüzde 25 = 0.25, ortalama = 0.68, maksimum = 8 GPU.**
>
> **Önemli:** Figürde 0 GPU çubuğu yoktur. Yukarıdaki kanonik tablo varsayılan
> `include_cpu_only=False` ile kurulduğundan 17.816 yalnızca-CPU işi (`num_gpu == 0`) zaten
> filtrelenmiştir ve burada çizilen her iş en az 0.01 GPU talep etmektedir. Çizelgeleme
> simülatörü de (Notebook 05) yalnızca GPU talepli işlerle çalışmaktadır. Bu dağılım,
> kümede küçük işlerin (≤ 1 GPU) hakimiyetini ve Notebook 05'teki Mid-Range (2 GPU) düğüm
> profilinin neden en anlamlı profil olduğunu açıklamaktadır.


In [ ]:
# ── 10. Inter-arrival time distribution ──────────────────────────────────────
df_sorted = job_df.sort_values("arrival_time").reset_index(drop=True)
inter_arrivals_all = df_sorted["arrival_time"].diff().dt.total_seconds().dropna()
_zero_frac = float((inter_arrivals_all == 0).mean())
# Same-second (zero inter-arrival) pairs cannot be placed on a log axis and
# are reported separately rather than silently dropped from the figure.
inter_arrivals = inter_arrivals_all[inter_arrivals_all > 0]

fig, ax = plt.subplots(figsize=(10, 5))
# Log-spaced bins, not np.histogram's linear default: with bins=100 linear
# but the axis itself log-scaled, the first (smallest) bin's fixed linear
# width covered a large share of the visually-compressed low end, making
# "peak occurs at 1-3s" an artifact of bin width rather than the data
# (figures_tables-9).
_log_bins = np.logspace(np.log10(inter_arrivals.min()), np.log10(inter_arrivals.max()), 40)
ax.hist(inter_arrivals, bins=_log_bins, color="mediumpurple", edgecolor="white", linewidth=0.3)
ax.set_xscale("log")
ax.set_xlabel("Inter-Arrival Time (seconds, log scale)")
ax.set_ylabel("Frequency")
ax.set_title("Inter-Arrival Time Distribution")
plt.tight_layout()
plt.show()
print(f"[Inter-arrival] {_zero_frac:.1%} of consecutive job pairs arrived in the same second "
      f"(excluded above; not visualizable on a log time axis).")

> **Şekil 5 — İşler Arası Geliş Aralığı Dağılımı (Log Ölçek):** X ekseni geliş
> aralığını log ölçekte saniye cinsinden, Y ekseni frekansı göstermektedir. Histogram,
> gözlenen aralığı (1 s – 237 s) kapsayan 40 log-aralıklı bin kullanmaktadır.
> - En yüksek bin **[1.00, 1.15) s: 12.110 çift** — en sık görülen durum bir saniyelik
>   aralıktır; 1 veya 2 saniyelik aralıklar toplamda 18.917 çift oluşturmaktadır.
> - 3 s ile 9.4 s arasındaki altı dolu bin toplam 18.105 çift, en yükseği ise 4.222 çift
>   içermektedir.
> - 10–100 saniye aralığında sayımlar genel olarak azalmaktadır: [10.8, 12.5) s binindeki
>   3.258 çiftten [88.8, 102.2) s binindeki 104 çifte iner.
> - 100 saniye ve üzerinde yalnızca 162 çift vardır ve gözlenen en büyük aralık 237 saniyedir.
> - **Aynı saniye içinde gelen ardışık işler (tüm çiftlerin %21,6'sı)** figüre dâhil
>   edilmemiştir (sıfır değeri log eksene yerleştirilemez); bu oran yukarıdaki hücre
>   tarafından ayrıca raporlanmaktadır.
>
> **Yorumu:** Dağılım üssel (exponential) değil, ağır-kuyruk yapısındadır. Bu,
> **homojen Poisson sürecinin (M/M/1) bu iş yükü için yetersiz** olduğunu kanıtlar.
> Ardışık işlerin beşte birinin aynı saniye içinde, geri kalanının büyük bölümünün ise
> birkaç saniye arayla geldiği burst dönemleri, kuyruk gecikmelerini belirlenimci değil
> olasılıksal (probabilistic) bir niteliğe taşır ve SJF'nin FIFO'ya karşı avantajını
> dinamik kılmaktadır.


In [ ]:
# ── 11. Day × Hour arrival heatmap ───────────────────────────────────────────
df_heat = job_df.copy()
df_heat["hour"] = df_heat["arrival_time"].dt.hour
# The trace starts at the epoch (1970-01-01), so .dt.dayofweek encodes where
# the trace happens to sit on the calendar, not a real weekday -- and it folds
# trace-day 0 and trace-day 7 into one row (11,106 + 6,571 = 17,677 jobs, the
# whole dayofweek==3 bucket). src/feature_engineering.py already uses a
# trace-day counter; this figure now matches it.
# For the same reason the x axis is not clock time: submit_time starts at 0, so
# the hour is a 24h phase whose absolute alignment the trace release never
# discloses. The label must not claim local or UTC time -- a reader would then
# take "hour 9" off this figure as 9am.
_t0_heat = df_heat["arrival_time"].min()
df_heat["trace_day"] = ((df_heat["arrival_time"] - _t0_heat).dt.total_seconds() // 86400).astype(int)
_n_days = int(df_heat["trace_day"].max()) + 1

heatmap_data = (
    df_heat.groupby(["trace_day", "hour"])["job_id"]
    .count()
    .unstack(fill_value=0)
    .reindex(index=range(_n_days), columns=range(24), fill_value=0)
)

# The last trace day is partial (the trace ends mid-afternoon), so its unreached
# hours are masked rather than drawn as genuine zero-arrival cells.
_hours_seen = df_heat.groupby("trace_day")["hour"].max()
_mask = pd.DataFrame(False, index=heatmap_data.index, columns=heatmap_data.columns)
for _d, _h_max in _hours_seen.items():
    _mask.loc[_d, [h for h in heatmap_data.columns if h > _h_max]] = True

fig, ax = plt.subplots(figsize=(14, 5))
# linewidths draws a grid over EVERY cell including the masked ones, so the
# blank block for the hours the trace never reached was crossed by stray grey
# lines that read as empty-but-real cells. The masked area is given a distinct
# flat colour instead, which separates "not measured" from the colormap's
# lowest value (a pale yellow that white was easily mistaken for).
sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap="YlGnBu",
    linewidths=0.3,
    mask=_mask,
    cbar_kws={"label": "Number of Jobs"},
)
ax.set_facecolor("0.85")
ax.set_xlabel("Hour Index within Trace Day (24h phase, not wall-clock time)")
ax.set_ylabel("Trace Day Index")
ax.set_title("Job Arrival Heatmap — Trace Day × Hour-of-Day")
plt.tight_layout()
plt.show()

> **Şekil 6 — İş Geliş Isı Haritası (İz-Günü × Günün Saati):** X ekseni
> günün saatini (0–23), Y ekseni iz-göreli gün indeksini (0 = izin ilk günü; iz ~7,7 gün sürüyor),
> renk yoğunluğu o bloktaki iş sayısını göstermektedir (sarı ~200 → koyu mavi ~1.200+).
>
> **Görsel örüntüler:** Gündüz saatleri, ısı haritasının tüm satırlarında gece saatlerine göre belirgin biçimde daha yoğun iş gönderimi gösteriyor — net bir gün-içi (diurnal) örüntü. Yoğunluk izin bir gününden diğerine de değişiyor, ama **bu, Pazartesi-Pazar haftalık bir örüntü olarak OKUNMAMALI** (aşağıdaki nota bakınız).
>
> **Saat Ekseni Notu:** `hour_of_day` özelliği, izin `submit_time` değerinin — iz başlangıcında sıfırdan başlayan bir ofsetin — Unix epoch'undan itibaren geçen saniye gibi okunmasıyla elde edilen `arrival_time` üzerinden türetilmiştir. Yayın, izin gerçekte ne zaman toplandığını açıklamadığından bu saatler **ne UTC duvar saatidir ne de kümenin yerel saati**: mutlak hizası bilinmeyen 24 saatlik bir fazdır ve bu eksendeki "saat 9", sabah 9 olarak okunamaz. **Göreli gün-içi (diurnal) örüntü** — bir gün içinde hangi saatlerin diğerlerine göre yoğun olduğu — tamamen geçerliliğini korumaktadır; bilinmeyen yalnızca mutlak hizadır.
>
> **Gün-indeksi tanımı (düzeltildi):** Bu defterin önceki bir sürümü `day_of_week`'i `arrival_time.dt.dayofweek` (0 = Pazartesi, ISO-8601) olarak tanımlıyor ve bu ısı haritasını takvim-günü örüntüsü (ör. "Perşembe zirvesi") gösteriyormuş gibi okuyordu. Bu okuma geçerli değil: Alibaba PAI izinin genel yayını gerçek toplama tarihini açıklamıyor ve iz yalnızca ~7,7 gün sürüyor, yani gerçek bir takvim-günü kodlaması izin ilk ve sekizinci gününü AYNI değere düşürür — açıklanmamış başlangıç tarihi yüzünden doğrulanamaz, ve kronolojik eğitim/test bölünmesi için bir sızıntı riski taşır, çünkü o paylaşılan değerin eğitimdeki TÜM örnekleri izin ilk gününden, test'teki TÜM örnekleri ise son gününden gelir (bilimsel denetimdeki `leakage-1`). `day_of_week` artık mutlak bir iz-günü sayacı (0, 1, 2, ...); bu analizde hiçbir yerde "Pazartesi" ya da "Perşembe" olarak okunmamalıdır.

## Özet

Bu notebook'ta gerçekleştirilen keşifsel veri analizi (EDA), tezin tüm sonraki metodolojik
kararlarını nesnel verilerle temellendiren kritik bulgular ortaya çıkarmıştır:

- **Runtime dağılımı — Ağır kuyruk:** Medyan birkaç dakika düzeyindeyken ağır-kuyruk
  yapısı saatlerce süren işlere uzanmaktadır. Bu dağılım, MSE yerine MAE/MdAE metriklerini,
  doğrusal yerine ağaç ve derin öğrenme modellerini zorunlu kılmaktadır.
- **GPU talebi — Heterojen:** 82.184 işin 80.209'u (%97,6) en fazla bir GPU talep etmekte,
  bunların 43.176'sı GPU paylaşımı yoluyla yalnızca bir GPU'nun kesrini istemektedir;
  1.975 iş ise 2 ile 8 arasında GPU talep etmektedir. İki büyüklük mertebesine yayılan bu
  aralık, simülatörde çoklu düğüm profillerini meşrulaştırmaktadır.
- **Zamansal örüntü — Diurnal cycle:** İz-göreli saat ekseni üzerinde belirgin bir
  gündüz/gece döngüsü, temporal özellik mühendisliğinin gerekliliğini ve kronolojik
  train/test bölmesinin zorunluluğunu kanıtlar. (Saat ekseni mutlak hizası bilinmeyen
  24 saatlik bir fazdır — bkz. Şekil 3 ve 6 notları — dolayısıyla döngü görelidir, gerçek
  bir saatteki mesai saatlerine bağlanamaz.)
- **Kullanıcı konsantrasyonu:** Küçük bir kullanıcı grubunun ağır iş yükü üretmesi,
  `user` özelliğinin One-Hot kodlamasının boyut patlamasına neden olduğunu açıklamaktadır.

Bu bulgular, sonraki notebook'lardaki her önemli kararın keyfi değil, **veri tarafından
güdüldüğünü** (data-driven) garanti altına almaktadır.
